In [ ]:
# =========================================================
# 0. GOOGLE DRIVE SETUP
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 1. IMPORTS
# =========================================================
import re
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# =========================================================
# 2. PATHS
# =========================================================
BASE_PATH = "/content/drive/MyDrive/GENAI_HACKS"

MESSAGES_PATH = f"{BASE_PATH}/relationship_ai_messages_3k.csv"
LABELS_PATH = f"{BASE_PATH}/relationship_ai_conversation_labels_3k.csv"

MODEL_PATH = f"{BASE_PATH}/relationship_signal_model.joblib"
METADATA_PATH = f"{BASE_PATH}/relationship_signal_model_metadata.json"
FEATURE_TABLE_PATH = f"{BASE_PATH}/relationship_signal_feature_table.csv"
PRED_SAMPLE_PATH = f"{BASE_PATH}/relationship_signal_predictions_sample.csv"

# =========================================================
# 3. LOAD DATA
# =========================================================
messages = pd.read_csv(MESSAGES_PATH)
labels = pd.read_csv(LABELS_PATH)

messages["timestamp"] = pd.to_datetime(messages["timestamp"])
messages = messages.sort_values(["conversation_id", "timestamp"]).reset_index(drop=True)

print("Messages shape:", messages.shape)
print("Labels shape:", labels.shape)
print(messages.head())
print(labels.head())

# =========================================================
# 4. PHRASE BANKS / SIGNAL BANKS
# =========================================================
VAGUE_PHRASES = [
    "sometime", "soon", "maybe later", "one day", "eventually",
    "we should hang", "we should do something", "proper date soon",
    "when life calms down", "later maybe"
]

SPECIFIC_PHRASES = [
    "tomorrow", "friday", "thursday", "saturday", "sunday",
    "at 7", "at 8", "after work", "this weekend", "free at",
    "around 7", "around 8", "coffee tomorrow", "drinks friday"
]

PRESSURE_PHRASES = [
    "why are you ignoring me", "answer me", "you owe me", "stop ignoring me",
    "left me on read", "you could reply", "don't play with me", "where did you go",
    "why didn't you answer", "you there?", "hello???", "dodging me"
]

AFFECTION_PHRASES = [
    "love", "obsessed", "favorite person", "different from everyone",
    "known you forever", "feel like home", "special", "attached already",
    "miss you", "thinking about you", "my person", "so into you"
]

REPAIR_PHRASES = [
    "sorry", "my bad", "that's on me", "i was overwhelmed",
    "i could've communicated better", "didn't mean to",
    "i get why you're annoyed", "i handled that badly"
]

POSITIVE_WORDS = [
    "good", "great", "cute", "sweet", "love", "funny", "excited",
    "happy", "amazing", "adorable", "nice", "beautiful"
]

NEGATIVE_WORDS = [
    "bad", "tired", "annoyed", "upset", "angry", "weird", "confused",
    "cold", "sad", "frustrated", "off", "dry"
]

# =========================================================
# 5. HELPER FUNCTIONS
# =========================================================
def contains_any(text: str, phrases: list) -> int:
    text = str(text).lower()
    return sum(1 for p in phrases if p in text)

def safe_mean(series):
    return float(series.mean()) if len(series) > 0 else 0.0

def safe_std(series):
    return float(series.std()) if len(series) > 1 else 0.0

def clip01(x):
    return max(0.0, min(1.0, float(x)))

# =========================================================
# 6. CONVERSATION-LEVEL FEATURE ENGINEERING
# =========================================================
def compute_conversation_features(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("timestamp").copy()
    group["message_text"] = group["message_text"].astype(str)
    group["word_count"] = group["message_text"].str.split().str.len()

    user_msgs = group[group["sender"] == "user"].copy()
    partner_msgs = group[group["sender"] == "partner"].copy()

    total_msgs = len(group)
    user_count = len(user_msgs)
    partner_count = len(partner_msgs)

    # Basic ratios
    partner_message_ratio = partner_count / total_msgs if total_msgs > 0 else 0.0

    avg_user_words = safe_mean(user_msgs["word_count"])
    avg_partner_words = safe_mean(partner_msgs["word_count"])
    length_ratio = (avg_partner_words / avg_user_words) if avg_user_words > 0 else 0.0

    # Initiation ratio: first message + any message after a long gap (>= 8 hours)
    initiators = []
    prev_time = None
    for _, row in group.iterrows():
        if prev_time is None:
            initiators.append(row["sender"])
        else:
            gap_hours = (row["timestamp"] - prev_time).total_seconds() / 3600
            if gap_hours >= 8:
                initiators.append(row["sender"])
        prev_time = row["timestamp"]

    partner_initiation_ratio = (
        initiators.count("partner") / len(initiators) if len(initiators) > 0 else 0.0
    )

    # Response times
    partner_reply_times = []
    user_reply_times = []

    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        curr = group.iloc[i]

        if prev["sender"] != curr["sender"]:
            delta_hours = (curr["timestamp"] - prev["timestamp"]).total_seconds() / 3600

            if curr["sender"] == "partner":
                partner_reply_times.append(delta_hours)
            else:
                user_reply_times.append(delta_hours)

    avg_partner_reply_hours = np.mean(partner_reply_times) if partner_reply_times else 0.0
    avg_user_reply_hours = np.mean(user_reply_times) if user_reply_times else 0.0
    partner_reply_std = np.std(partner_reply_times) if len(partner_reply_times) > 1 else 0.0

    # Reappearance cycles = gaps >= 2 days
    reappearance_cycles = 0
    for i in range(1, len(group)):
        gap_days = (group.iloc[i]["timestamp"] - group.iloc[i - 1]["timestamp"]).total_seconds() / 86400
        if gap_days >= 2:
            reappearance_cycles += 1

    # Decline in partner participation over time
    half = max(1, len(group) // 2)
    first_half = group.iloc[:half]
    second_half = group.iloc[half:]

    first_half_partner_msgs = (first_half["sender"] == "partner").sum()
    second_half_partner_msgs = (second_half["sender"] == "partner").sum()

    if first_half_partner_msgs > 0:
        partner_decline_ratio = (first_half_partner_msgs - second_half_partner_msgs) / first_half_partner_msgs
        partner_decline_ratio = max(0.0, partner_decline_ratio)
    else:
        partner_decline_ratio = 0.0

    # Short replies
    partner_short_reply_ratio = (
        partner_msgs["word_count"].le(2).mean() if partner_count > 0 else 0.0
    )

    # Phrase counts
    partner_text = partner_msgs["message_text"].astype(str).str.lower()

    vague_phrase_count = int(partner_text.apply(lambda x: contains_any(x, VAGUE_PHRASES)).sum())
    specific_phrase_count = int(partner_text.apply(lambda x: contains_any(x, SPECIFIC_PHRASES)).sum())
    pressure_phrase_count = int(partner_text.apply(lambda x: contains_any(x, PRESSURE_PHRASES)).sum())
    affection_count = int(partner_text.apply(lambda x: contains_any(x, AFFECTION_PHRASES)).sum())
    repair_count = int(partner_text.apply(lambda x: contains_any(x, REPAIR_PHRASES)).sum())

    positive_count = int(partner_text.apply(lambda x: sum(1 for w in POSITIVE_WORDS if w in x)).sum())
    negative_count = int(partner_text.apply(lambda x: sum(1 for w in NEGATIVE_WORDS if w in x)).sum())

    # Ending state
    unanswered_last_user_message = int(group.iloc[-1]["sender"] == "user")

    # Time span
    days_span = (group["timestamp"].max() - group["timestamp"].min()).total_seconds() / 86400

    # Full conversation text for TF-IDF
    conversation_text = " ".join(
        [f'{row["sender"]}: {row["message_text"]}' for _, row in group.iterrows()]
    )tion": "Reduce over-investment and wait for consistent effort before moving forward.",
    "suggested_response": "No worries \u2014 let me know w
    return pd.Series({
        "conversation_id": group["conversation_id"].iloc[0],
        "conversation_text": conversation_text,
        "total_messages": total_msgs,
        "user_messages": user_count,
        "partner_messages": partner_count,
        "partner_message_ratio": partner_message_ratio,
        "avg_user_words": avg_user_words,
        "avg_partner_words": avg_partner_words,
        "length_ratio": length_ratio,
        "partner_initiation_ratio": partner_initiation_ratio,
        "avg_partner_reply_hours": avg_partner_reply_hours,
        "avg_user_reply_hours": avg_user_reply_hours,
        "partner_reply_std": partner_reply_std,
        "reappearance_cycles": reappearance_cycles,
        "partner_decline_ratio": partner_decline_ratio,
        "partner_short_reply_ratio": partner_short_reply_ratio,
        "vague_phrase_count": vague_phrase_count,
        "specific_phrase_count": specific_phrase_count,
        "pressure_phrase_count": pressure_phrase_count,
        "affection_count": affection_count,
        "repair_count": repair_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "unanswered_last_user_message": unanswered_last_user_message,
        "days_span": days_span,
    })

# =========================================================
# 7. BUILD TRAINING TABLE
# =========================================================
conv_features = messages.groupby("conversation_id").apply(compute_conversation_features).reset_index(drop=True)

train_df = conv_features.merge(labels, on="conversation_id", how="inner")

print("Conversation feature table shape:", train_df.shape)
display(train_df.head())

# Save feature table too
train_df.to_csv(FEATURE_TABLE_PATH, index=False)
print(f"Saved feature table to: {FEATURE_TABLE_PATH}")

# =========================================================
# 8. DEFINE TARGETS + FEATURES
# =========================================================
TARGETS = [
    "effort_balance",
    "ghosting_probability",
    "breadcrumbing_risk",
    "lovebombing_risk",
    "boundary_violation_risk",
]

NUMERIC_FEATURES = [
    "total_messages",
    "user_messages",
    "partner_messages",
    "partner_message_ratio",
    "avg_user_words",
    "avg_partner_words",
    "length_ratio",
    "partner_initiation_ratio",
    "avg_partner_reply_hours",
    "avg_user_reply_hours",
    "partner_reply_std",
    "reappearance_cycles",
    "partner_decline_ratio",
    "partner_short_reply_ratio",
    "vague_phrase_count",
    "specific_phrase_count",
    "pressure_phrase_count",
    "affection_count",
    "repair_count",
    "positive_count",
    "negative_count",
    "unanswered_last_user_message",
    "days_span",
]

X_num = train_df[NUMERIC_FEATURES].fillna(0)
X_text = train_df["conversation_text"].fillna("")
y = train_df[TARGETS].copy()

# =========================================================
# 9. TEXT FEATURES
# =========================================================
tfidf = TfidfVectorizer(
    max_features=4000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    lowercase=True
)

X_text_tfidf = tfidf.fit_transform(X_text)

svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

X_text_svd = svd.fit_transform(X_text_tfidf)

X_final = np.hstack([X_num.values, X_text_svd])

print("Final training matrix shape:", X_final.shape)

# =========================================================
# 10. TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42
)

# =========================================================
# 11. TRAIN MODEL
# =========================================================
model = MultiOutputRegressor(
    ExtraTreesRegressor(
        n_estimators=250,
        random_state=42,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        n_jobs=-1
    )
)

model.fit(X_train, y_train)

# =========================================================
# 12. EVALUATE
# =========================================================
preds = model.predict(X_test)

print("\nModel Evaluation:")
for i, target in enumerate(TARGETS):
    mae = mean_absolute_error(y_test.iloc[:, i], preds[:, i])
    r2 = r2_score(y_test.iloc[:, i], preds[:, i])
    print(f"{target}: MAE={mae:.4f} | R2={r2:.4f}")

# Save a sample of predictions
pred_df = pd.DataFrame(preds, columns=[f"pred_{t}" for t in TARGETS])
pred_df["conversation_id"] = train_df.iloc[y_test.index]["conversation_id"].values
pred_df.to_csv(PRED_SAMPLE_PATH, index=False)
print(f"Saved sample predictions to: {PRED_SAMPLE_PATH}")

# =========================================================
# 13. SAVE MODEL BUNDLE
# =========================================================
bundle = {
    "model": model,
    "tfidf": tfidf,
    "svd": svd,
    "numeric_features": NUMERIC_FEATURES,
    "targets": TARGETS,
    "phrase_banks": {
        "vague_phrases": VAGUE_PHRASES,
        "specific_phrases": SPECIFIC_PHRASES,
        "pressure_phrases": PRESSURE_PHRASES,
        "affection_phrases": AFFECTION_PHRASES,
        "repair_phrases": REPAIR_PHRASES,
        "positive_words": POSITIVE_WORDS,
        "negative_words": NEGATIVE_WORDS,
    }
}

joblib.dump(bundle, MODEL_PATH)
print(f"Saved model bundle to: {MODEL_PATH}")

metadata = {
    "numeric_features": NUMERIC_FEATURES,
    "targets": TARGETS,
    "messages_path": MESSAGES_PATH,
    "labels_path": LABELS_PATH,
    "model_path": MODEL_PATH
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata to: {METADATA_PATH}")

# =========================================================
# 14. LIVE INFERENCE FUNCTIONS
# =========================================================
def extract_live_features(conversation):
    """
    conversation: list of dicts with keys:
      - sender
      - timestamp
      - text
    """
    df = pd.DataFrame(conversation).copy()
    df["conversation_id"] = "live_conv"
    df = df.rename(columns={"text": "message_text"})
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    feats = compute_conversation_features(df)
    return feats

def build_model_input_from_live_conversation(conversation, bundle):
    feats = extract_live_features(conversation)

    # numeric features
    x_num = feats[bundle["numeric_features"]].fillna(0).values.reshape(1, -1)

    # text features
    x_text = bundle["tfidf"].transform([feats["conversation_text"]])
    x_text_svd = bundle["svd"].transform(x_text)

    x_final = np.hstack([x_num, x_text_svd])
    return feats, x_final

def generate_recommendation(scores):
    effort = scores["effort_balance"]
    ghost = scores["ghosting_probability"]
    bread = scores["breadcrumbing_risk"]
    love = scores["lovebombing_risk"]
    boundary = scores["boundary_violation_risk"]

    flags = []

    if effort < 0.35:
        flags.append("low reciprocal effort")
    if ghost > 0.65:
        flags.append("rising ghosting risk")
    if bread > 0.60:
        flags.append("breadcrumbing pattern")
    if love > 0.65:
        flags.append("intense early affection / love bombing")
    if boundary > 0.60:
        flags.append("boundary-pushing language")

    if not flags:
        summary = "No major red flags detected."
    else:
        summary = "Detected signals: " + ", ".join(flags) + "."

    if boundary > 0.60:
        action = "Set a clear boundary and do not reward pressure with immediate over-explaining."
    elif bread > 0.60 or ghost > 0.65:
        action = "Reduce over-investment and wait for specific, consistent effort before moving forward."
    elif love > 0.65:
        action = "Proceed slowly and look for consistency over time, not just intensity."
    elif effort < 0.35:
        action = "Stop carrying the interaction alone and look for reciprocal engagement."
    else:
        action = "Conversation looks reasonably balanced so far. Keep watching for consistency over time."

    return {
        "summary": summary,
        "recommended_action": action
    }

def analyze_live_conversation(conversation, bundle):
    feats, x_final = build_model_input_from_live_conversation(conversation, bundle)
    pred = bundle["model"].predict(x_final)[0]

    scores = {
        bundle["targets"][i]: clip01(pred[i])
        for i in range(len(bundle["targets"]))
    }

    recommendation = generate_recommendation(scores)

    return {
        "scores": scores,
        "features": {
            k: float(feats[k]) if isinstance(feats[k], (np.floating, float, int, np.integer)) else feats[k]
            for k in bundle["numeric_features"]
        },
        "recommendation": recommendation
    }

# =========================================================
# 15. TEST LIVE INFERENCE
# =========================================================
sample_conversation = [
    {"sender": "user", "timestamp": "2026-03-14 20:01:00", "text": "hey how was your day"},
    {"sender": "partner", "timestamp": "2026-03-14 20:05:00", "text": "pretty good actually just got home from work"},
    {"sender": "user", "timestamp": "2026-03-14 20:06:00", "text": "nice haha what do you do again"},
    {"sender": "partner", "timestamp": "2026-03-16 19:30:00", "text": "sorry just seeing this now lol"},
    {"sender": "user", "timestamp": "2026-03-16 19:31:00", "text": "all good haha"},
    {"sender": "partner", "timestamp": "2026-03-20 21:00:00", "text": "hey stranger we should hang sometime"},
]

live_result = analyze_live_conversation(sample_conversation, bundle)

print("\nLive Inference Example:")
print(json.dumps(live_result, indent=2))

Mounted at /content/drive
Messages shape: (70811, 5)
Labels shape: (3000, 7)
  conversation_id message_id   sender           timestamp  \
0          conv_1         m1     user 2025-02-14 12:00:00   
1          conv_1         m2  partner 2025-02-14 12:10:00   
2          conv_1         m3     user 2025-02-15 12:10:00   
3          conv_1         m4  partner 2025-02-15 12:40:00   
4          conv_1         m5     user 2025-02-15 13:40:00   

                                        message_text  
0  okay important question: are you funny or just...  
1                                          what's up  
2                                              hi :)  
3                                i just left the gym  
4                              that's actually funny  
  conversation_id     scenario  effort_balance  ghosting_probability  \
0          conv_1      healthy           0.601                 0.143   
1          conv_2  lovebombing           0.262                 0.270   
2         

/tmp/ipykernel_606/3854743904.py:247: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  conv_features = messages.groupby("conversation_id").apply(compute_conversation_features).reset_index(drop=True)


,conversation_id,conversation_text,total_messages,user_messages,partner_messages,partner_message_ratio,avg_user_words,avg_partner_words,length_ratio,partner_initiation_ratio,...,positive_count,negative_count,unanswered_last_user_message,days_span,scenario,effort_balance,ghosting_probability,breadcrumbing_risk,lovebombing_risk,boundary_violation_risk
0,conv_1,user: okay important question: are you funny o...,20,10,10,0.500000,4.600000,2.800000,0.608696,0.333333,...,0,1,0,2.259028,healthy,0.601,0.143,0.343,0.154,0.069
1,conv_10,user: hey partner: not me blushing over text u...,19,10,9,0.473684,3.600000,5.444444,1.512346,0.750000,...,3,1,1,3.589583,breadcrumbing,0.344,0.175,0.700,0.156,0.057
2,conv_100,user: heyy partner: i feel like you'd bully me...,21,11,10,0.476190,3.545455,4.500000,1.269231,0.250000,...,1,0,1,3.498611,breadcrumbing,0.808,0.247,0.700,0.168,0.192
3,conv_1000,user: random question but coffee or matcha par...,35,18,17,0.485714,4.722222,3.764706,0.797232,0.571429,...,3,1,1,6.942361,boundary_issue,0.405,0.165,0.205,0.069,1.000
4,conv_1001,user: what's up partner: you left me on read u...,29,15,14,0.482759,3.400000,4.785714,1.407563,0.250000,...,3,0,1,3.711806,boundary_issue,0.588,0.327,0.068,0.197,1.000


Saved feature table to: /content/drive/MyDrive/GENAI_HACKS/relationship_signal_feature_table.csv
Final training matrix shape: (3000, 123)

Model Evaluation:
effort_balance: MAE=0.1769 | R2=0.0043
ghosting_probability: MAE=0.0722 | R2=0.8453
breadcrumbing_risk: MAE=0.0944 | R2=0.6631
lovebombing_risk: MAE=0.0441 | R2=0.9700
boundary_violation_risk: MAE=0.0461 | R2=0.9667
Saved sample predictions to: /content/drive/MyDrive/GENAI_HACKS/relationship_signal_predictions_sample.csv
Saved model bundle to: /content/drive/MyDrive/GENAI_HACKS/relationship_signal_model.joblib
Saved metadata to: /content/drive/MyDrive/GENAI_HACKS/relationship_signal_model_metadata.json


/tmp/ipykernel_606/3854743904.py:425: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_num = feats[bundle["numeric_features"]].fillna(0).values.reshape(1, -1)



Live Inference Example:
{
  "scores": {
    "effort_balance": 0.5486546666666667,
    "ghosting_probability": 0.7894220000000044,
    "breadcrumbing_risk": 0.23879266666666701,
    "lovebombing_risk": 0.09181533333333326,
    "boundary_violation_risk": 0.10233066666666671
  },
  "features": {
    "total_messages": 6.0,
    "user_messages": 3.0,
    "partner_messages": 3.0,
    "partner_message_ratio": 0.5,
    "avg_user_words": 5.0,
    "avg_partner_words": 6.666666666666667,
    "length_ratio": 1.3333333333333335,
    "partner_initiation_ratio": 0.6666666666666666,
    "avg_partner_reply_hours": 48.31666666666666,
    "avg_user_reply_hours": 0.016666666666666666,
    "partner_reply_std": 39.77546936194421,
    "reappearance_cycles": 1.0,
    "partner_decline_ratio": 0.0,
    "partner_short_reply_ratio": 0.0,
    "vague_phrase_count": 2.0,
    "specific_phrase_count": 0.0,
    "pressure_phrase_count": 0.0,
    "affection_count": 0.0,
    "repair_count": 1.0,
    "positive_count": 1.0,

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib

# =========================================================
# LOAD MODEL BUNDLE
# =========================================================
BASE_PATH = "/content/drive/MyDrive/GENAI_HACKS"
MODEL_PATH = f"{BASE_PATH}/relationship_signal_model.joblib"

bundle = joblib.load(MODEL_PATH)

# =========================================================
# HELPER: CLIP
# =========================================================
def clip01(x):
    return max(0.0, min(1.0, float(x)))

# =========================================================
# RULE-BASED EFFORT SCORE
# =========================================================
def compute_rule_based_effort(feats):
    partner_ratio = float(feats["partner_message_ratio"])
    length_ratio = float(feats["length_ratio"])
    initiation_ratio = float(feats["partner_initiation_ratio"])
    partner_reply_std = float(feats["partner_reply_std"])
    vague_count = float(feats["vague_phrase_count"])
    specific_count = float(feats["specific_phrase_count"])

    # closer to 50/50 is better
    ratio_score = 1 - abs(partner_ratio - 0.5) / 0.5
    ratio_score = clip01(ratio_score)

    # closer message lengths is better
    if length_ratio > 1:
        length_score = 1 / length_ratio
    else:
        length_score = length_ratio
    length_score = clip01(length_score)

    # closer initiation split is better
    init_score = 1 - abs(initiation_ratio - 0.5) / 0.5
    init_score = clip01(init_score)

    # lower std means more consistent replying
    response_consistency = 1 - min(partner_reply_std / 48.0, 1.0)

    # specific plans are better than vague plans
    plan_total = vague_count + specific_count
    if plan_total > 0:
        planning_score = specific_count / plan_total
    else:
        planning_score = 0.5

    effort_score = (
        0.30 * ratio_score +
        0.20 * length_score +
        0.20 * init_score +
        0.15 * response_consistency +
        0.15 * planning_score
    )

    return round(clip01(effort_score), 4)

# =========================================================
# RULE-BASED BREADCRUMB BOOST
# =========================================================
def compute_breadcrumb_rule_boost(feats, conversation):
    vague_count = float(feats["vague_phrase_count"])
    specific_count = float(feats["specific_phrase_count"])
    reappearance_cycles = float(feats["reappearance_cycles"])
    avg_partner_reply_hours = float(feats["avg_partner_reply_hours"])
    partner_ratio = float(feats["partner_message_ratio"])

    texts = " || ".join([m["text"].lower() for m in conversation])

    hey_stranger_flag = 1.0 if (
        "hey stranger" in texts or
        "missed your face" in texts or
        "randomly thought about you" in texts or
        "sorry i've been busy" in texts or
        "just seeing this now" in texts
    ) else 0.0

    if vague_count > 0 and specific_count == 0:
        vague_without_specific = 1.0
    elif vague_count > specific_count:
        vague_without_specific = 0.6
    else:
        vague_without_specific = 0.0

    slow_reentry = min(avg_partner_reply_hours / 72.0, 1.0)
    low_follow_through = 1.0 - min(specific_count / max(vague_count, 1.0), 1.0)
    effort_penalty = 1.0 - min(partner_ratio / 0.5, 1.0) if partner_ratio < 0.5 else 0.0
    reappearance_score = min(reappearance_cycles / 2.0, 1.0)

    rule_breadcrumb = (
        0.28 * reappearance_score +
        0.24 * vague_without_specific +
        0.18 * hey_stranger_flag +
        0.15 * slow_reentry +
        0.15 * low_follow_through +
        0.10 * effort_penalty
    )

    return round(clip01(rule_breadcrumb), 4)

# =========================================================
# SIGNALS DETECTED
# =========================================================
def generate_signals_detected(feats, scores, conversation):
    signals = []

    if float(feats["avg_partner_reply_hours"]) > 24:
        signals.append("long response gaps")

    if float(feats["reappearance_cycles"]) >= 1:
        signals.append("reappearance after silence")

    if float(feats["vague_phrase_count"]) > 0 and float(feats["specific_phrase_count"]) == 0:
        signals.append("vague planning without specific follow-through")

    texts = " || ".join([m["text"].lower() for m in conversation])
    if "hey stranger" in texts or "just seeing this now" in texts or "sorry i've been busy" in texts:
        signals.append("warm re-entry after distance")

    if float(feats["pressure_phrase_count"]) > 0:
        signals.append("pressure or guilt-based language")

    if float(feats["affection_count"]) > 1 and scores["lovebombing_risk"] > 0.6:
        signals.append("intense early affection")

    if scores["effort_balance"] < 0.35:
        signals.append("one-sided conversational effort")

    if not signals:
        signals.append("no major behavioral red flags detected yet")

    # remove duplicates while preserving order
    deduped = []
    for s in signals:
        if s not in deduped:
            deduped.append(s)

    return deduped

# =========================================================
# RECOMMENDATION + SUGGESTED RESPONSE
# =========================================================
def generate_recommendation(scores):
    effort = scores["effort_balance"]
    ghost = scores["ghosting_probability"]
    bread = scores["breadcrumbing_risk"]
    love = scores["lovebombing_risk"]
    boundary = scores["boundary_violation_risk"]

    if boundary > 0.60:
        return {
            "action": "Set a clear boundary and do not reward pressure with immediate over-explaining.",
            "suggested_response": "I’m happy to talk, but I’m not okay with being spoken to like that."
        }

    if bread > 0.60:
        return {
            "action": "Look for concrete follow-through, not just periodic re-entry or vague interest.",
            "suggested_response": "I’d be open to meeting, but I prefer making specific plans instead of ‘sometime.’"
        }

    if ghost > 0.65:
        return {
            "action": "Reduce over-investment and wait for consistent effort before moving forward.",
            "suggested_response": "No worries — let me know when you actually know your availability."
        }

    if love > 0.65:
        return {
            "action": "Proceed slowly and judge consistency over time, not just intensity.",
            "suggested_response": "I like getting to know you too — I just prefer taking things at a steady pace."
        }

    if effort < 0.35:
        return {
            "action": "Stop carrying the interaction alone and wait for reciprocal effort.",
            "suggested_response": "I’m happy to keep talking, but I’m looking for mutual effort."
        }

    return {
        "action": "Conversation looks reasonably balanced so far. Keep watching for consistency over time.",
        "suggested_response": "I’m enjoying this so far — let’s keep seeing how the energy feels."
    }

# =========================================================
# FEATURE EXTRACTION FOR LIVE CONVO
# =========================================================
def compute_conversation_features(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("timestamp").copy()
    group["message_text"] = group["message_text"].astype(str)
    group["word_count"] = group["message_text"].str.split().str.len()

    user_msgs = group[group["sender"] == "user"].copy()
    partner_msgs = group[group["sender"] == "partner"].copy()

    total_msgs = len(group)
    user_count = len(user_msgs)
    partner_count = len(partner_msgs)

    partner_message_ratio = partner_count / total_msgs if total_msgs > 0 else 0.0

    avg_user_words = float(user_msgs["word_count"].mean()) if user_count > 0 else 0.0
    avg_partner_words = float(partner_msgs["word_count"].mean()) if partner_count > 0 else 0.0
    length_ratio = (avg_partner_words / avg_user_words) if avg_user_words > 0 else 0.0

    initiators = []
    prev_time = None
    for _, row in group.iterrows():
        if prev_time is None:
            initiators.append(row["sender"])
        else:
            gap_hours = (row["timestamp"] - prev_time).total_seconds() / 3600
            if gap_hours >= 8:
                initiators.append(row["sender"])
        prev_time = row["timestamp"]

    partner_initiation_ratio = (
        initiators.count("partner") / len(initiators) if len(initiators) > 0 else 0.0
    )

    partner_reply_times = []
    user_reply_times = []

    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        curr = group.iloc[i]
        if prev["sender"] != curr["sender"]:
            delta_hours = (curr["timestamp"] - prev["timestamp"]).total_seconds() / 3600
            if curr["sender"] == "partner":
                partner_reply_times.append(delta_hours)
            else:
                user_reply_times.append(delta_hours)

    avg_partner_reply_hours = np.mean(partner_reply_times) if partner_reply_times else 0.0
    avg_user_reply_hours = np.mean(user_reply_times) if user_reply_times else 0.0
    partner_reply_std = np.std(partner_reply_times) if len(partner_reply_times) > 1 else 0.0

    reappearance_cycles = 0
    for i in range(1, len(group)):
        gap_days = (group.iloc[i]["timestamp"] - group.iloc[i - 1]["timestamp"]).total_seconds() / 86400
        if gap_days >= 2:
            reappearance_cycles += 1

    half = max(1, len(group) // 2)
    first_half = group.iloc[:half]
    second_half = group.iloc[half:]

    first_half_partner_msgs = (first_half["sender"] == "partner").sum()
    second_half_partner_msgs = (second_half["sender"] == "partner").sum()

    if first_half_partner_msgs > 0:
        partner_decline_ratio = (first_half_partner_msgs - second_half_partner_msgs) / first_half_partner_msgs
        partner_decline_ratio = max(0.0, partner_decline_ratio)
    else:
        partner_decline_ratio = 0.0

    partner_short_reply_ratio = (
        partner_msgs["word_count"].le(2).mean() if partner_count > 0 else 0.0
    )

    partner_text = partner_msgs["message_text"].astype(str).str.lower()

    vague_phrases = bundle["phrase_banks"]["vague_phrases"]
    specific_phrases = bundle["phrase_banks"]["specific_phrases"]
    pressure_phrases = bundle["phrase_banks"]["pressure_phrases"]
    affection_phrases = bundle["phrase_banks"]["affection_phrases"]
    repair_phrases = bundle["phrase_banks"]["repair_phrases"]
    positive_words = bundle["phrase_banks"]["positive_words"]
    negative_words = bundle["phrase_banks"]["negative_words"]

    def contains_any(text, phrases):
        text = str(text).lower()
        return sum(1 for p in phrases if p in text)

    vague_phrase_count = int(partner_text.apply(lambda x: contains_any(x, vague_phrases)).sum())
    specific_phrase_count = int(partner_text.apply(lambda x: contains_any(x, specific_phrases)).sum())
    pressure_phrase_count = int(partner_text.apply(lambda x: contains_any(x, pressure_phrases)).sum())
    affection_count = int(partner_text.apply(lambda x: contains_any(x, affection_phrases)).sum())
    repair_count = int(partner_text.apply(lambda x: contains_any(x, repair_phrases)).sum())

    positive_count = int(partner_text.apply(lambda x: sum(1 for w in positive_words if w in x)).sum())
    negative_count = int(partner_text.apply(lambda x: sum(1 for w in negative_words if w in x)).sum())

    unanswered_last_user_message = int(group.iloc[-1]["sender"] == "user")
    days_span = (group["timestamp"].max() - group["timestamp"].min()).total_seconds() / 86400

    conversation_text = " ".join(
        [f'{row["sender"]}: {row["message_text"]}' for _, row in group.iterrows()]
    )

    return pd.Series({
        "conversation_id": group["conversation_id"].iloc[0],
        "conversation_text": conversation_text,
        "total_messages": total_msgs,
        "user_messages": user_count,
        "partner_messages": partner_count,
        "partner_message_ratio": partner_message_ratio,
        "avg_user_words": avg_user_words,
        "avg_partner_words": avg_partner_words,
        "length_ratio": length_ratio,
        "partner_initiation_ratio": partner_initiation_ratio,
        "avg_partner_reply_hours": avg_partner_reply_hours,
        "avg_user_reply_hours": avg_user_reply_hours,
        "partner_reply_std": partner_reply_std,
        "reappearance_cycles": reappearance_cycles,
        "partner_decline_ratio": partner_decline_ratio,
        "partner_short_reply_ratio": partner_short_reply_ratio,
        "vague_phrase_count": vague_phrase_count,
        "specific_phrase_count": specific_phrase_count,
        "pressure_phrase_count": pressure_phrase_count,
        "affection_count": affection_count,
        "repair_count": repair_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "unanswered_last_user_message": unanswered_last_user_message,
        "days_span": days_span,
    })

def extract_live_features(conversation):
    df = pd.DataFrame(conversation).copy()
    df["conversation_id"] = "live_conv"
    df = df.rename(columns={"text": "message_text"})
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    feats = compute_conversation_features(df)
    return feats

def build_model_input_from_live_conversation(conversation, bundle):
    feats = extract_live_features(conversation)
    x_num = feats[bundle["numeric_features"]].astype(float).values.reshape(1, -1)
    x_text = bundle["tfidf"].transform([feats["conversation_text"]])
    x_text_svd = bundle["svd"].transform(x_text)
    x_final = np.hstack([x_num, x_text_svd])
    return feats, x_final

# =========================================================
# FINAL HYBRID ANALYSIS
# =========================================================
def analyze_live_conversation_hybrid(conversation, bundle):
    feats, x_final = build_model_input_from_live_conversation(conversation, bundle)
    pred = bundle["model"].predict(x_final)[0]

    model_scores = {
        bundle["targets"][i]: clip01(pred[i])
        for i in range(len(bundle["targets"]))
    }

    # override effort with rule-based
    effort_rule = compute_rule_based_effort(feats)

    # boost breadcrumbing
    breadcrumb_rule = compute_breadcrumb_rule_boost(feats, conversation)
    breadcrumb_hybrid = max(
        model_scores["breadcrumbing_risk"],
        0.55 * model_scores["breadcrumbing_risk"] + 0.45 * breadcrumb_rule
    )
    breadcrumb_hybrid = clip01(breadcrumb_hybrid)

    scores = {
        "effort_balance": effort_rule,
        "ghosting_probability": model_scores["ghosting_probability"],
        "breadcrumbing_risk": breadcrumb_hybrid,
        "lovebombing_risk": model_scores["lovebombing_risk"],
        "boundary_violation_risk": model_scores["boundary_violation_risk"],
    }

    signals_detected = generate_signals_detected(feats, scores, conversation)
    recommendation = generate_recommendation(scores)

    # confidence based on history length
    total_messages = float(feats["total_messages"])
    if total_messages < 10:
        confidence_level = "low_history"
    elif total_messages < 30:
        confidence_level = "moderate_history"
    else:
        confidence_level = "high_history"

    return {
        "scores": scores,
        "signals_detected": signals_detected,
        "confidence_level": confidence_level,
        "recommendation": recommendation,
        "model_scores_raw": model_scores,
        "rule_scores": {
            "effort_balance_rule": effort_rule,
            "breadcrumbing_rule_boost": breadcrumb_rule
        },
        "features": {
            k: float(feats[k]) if isinstance(feats[k], (np.floating, float, int, np.integer)) else feats[k]
            for k in bundle["numeric_features"]
        }
    }

# =========================================================
# EXAMPLE TEST
# =========================================================
sample_conversation = [
    {"sender": "user", "timestamp": "2026-03-14 20:01:00", "text": "hey how was your day"},
    {"sender": "partner", "timestamp": "2026-03-14 20:05:00", "text": "pretty good actually just got home from work"},
    {"sender": "user", "timestamp": "2026-03-14 20:06:00", "text": "nice haha what do you do again"},
    {"sender": "partner", "timestamp": "2026-03-16 19:30:00", "text": "sorry just seeing this now lol"},
    {"sender": "user", "timestamp": "2026-03-16 19:31:00", "text": "all good haha"},
    {"sender": "partner", "timestamp": "2026-03-20 21:00:00", "text": "hey stranger we should hang sometime"},
]

result = analyze_live_conversation_hybrid(sample_conversation, bundle)
print(json.dumps(result, indent=2))

{
  "scores": {
    "effort_balance": 0.609,
    "ghosting_probability": 0.7894220000000044,
    "breadcrumbing_risk": 0.49615096666666686,
    "lovebombing_risk": 0.09181533333333326,
    "boundary_violation_risk": 0.10233066666666671
  },
  "signals_detected": [
    "long response gaps",
    "reappearance after silence",
    "vague planning without specific follow-through",
    "warm re-entry after distance"
  ],
  "confidence_level": "low_history",
  "recommendation": {
    "action": "Reduce over-investment and wait for consistent effort before moving forward.",
    "suggested_response": "No worries \u2014 let me know when you actually know your availability."
  },
  "model_scores_raw": {
    "effort_balance": 0.5486546666666667,
    "ghosting_probability": 0.7894220000000044,
    "breadcrumbing_risk": 0.23879266666666701,
    "lovebombing_risk": 0.09181533333333326,
    "boundary_violation_risk": 0.10233066666666671
  },
  "rule_scores": {
    "effort_balance_rule": 0.609,
    "bre